# Feature Engineering Pipeline for Delivery Prediction

This notebook prepares order-level features for predicting late deliveries.

Final output: 96,462 orders with 40 columns (1 target + 39 features)

Key feature groups:
- Temporal features: day of week, hour, month
- Payment features: payment types, installments, counts
- Product features: weights, volumes, item counts
- Seller features: seller counts, state matching flags
- Distance features: customer-seller distances
- Historical features: seller past performance

All features are leakage-free and available at prediction time.

## 1. Data Loading and Preparation
Load base tables from gold and silver layers for feature engineering.

In [0]:
gold_fact_orders = spark.read.table("revenue_operations.gold.fact_orders")
silver_orders = spark.read.table("revenue_operations.silver.orders")
gold_fact_order_items = spark.read.table("revenue_operations.gold.fact_order_items")

In [0]:
# Add order_approved_at from silver layer (needed for temporal features and historical calculations)
gold_fact_orders = gold_fact_orders.join(silver_orders.select("order_id", "order_approved_at"), 
                                         on = "order_id", how = "left")

## 2. Building Order-Level Base Table

Combine data from multiple sources to create the base feature table:
- Clean nulls from orders
- Add temporal features (day of week, hour, month)
- Aggregate order items to order level
- Join customer location data
- Aggregate and one-hot encode payment types

In [0]:
from pyspark.sql import functions as F

# Check for null values in all columns
print("Total rows in silver orders: " , gold_fact_orders.count())

for col in gold_fact_orders.columns:
    print(f"Number of null values in {col}: {gold_fact_orders.filter(F.col(col).isNull()).count()}")

# Remove rows with null delivery dates or approval timestamps
gold_fact_orders_clean = gold_fact_orders.dropna(how="any", subset=["delivery_date", "order_approved_at"])

print("Total rows in cleaned silver order with no null delivery dates and approved dates: ", gold_fact_orders_clean.count())

In [0]:
# Extract day of week, hour, and month from purchase timestamp
gold_fact_orders_clean = gold_fact_orders_clean.withColumn(
    "purchase_day_of_week", F.dayofweek(F.col("order_purchase_timestamp"))
).withColumn(
    "purchase_hour", F.hour(F.col("order_purchase_timestamp"))
).withColumn(
    "purchase_month", F.month(F.col("order_purchase_timestamp"))
)
display(gold_fact_orders_clean)

In [0]:
# Inspect order items table structure and row counts
print("Total rows in gold fact order items: ", gold_fact_order_items.count())
gold_fact_order_items.printSchema()
print("Total Number of distinct order_id: ", gold_fact_order_items.select("order_id").distinct().count())

In [0]:
from pyspark.sql import functions as F

# Sum product costs and quantities across all items per order
gold_fact_order_items = gold_fact_order_items.groupBy("order_id").agg(
    F.sum("product_subtotal").alias("product_subtotal"),
    F.sum("freight_total").alias("freight_total"),
    F.sum("quantity").alias("quantity"),
    F.sum("line_total").alias("line_total")
)


gold_fact_order_items.count()

In [0]:
# Left join aggregated order items back to orders
gold_orders_order_items = gold_fact_orders_clean.join(
    gold_fact_order_items.select("order_id", "product_subtotal", "freight_total", "quantity"),
    on="order_id", how="left"
)
print("Row Count :", gold_orders_order_items.count())
gold_orders_order_items.printSchema()

In [0]:
# Load customer data for geographic features
silver_customers = spark.read.table("revenue_operations.silver.customers")

In [0]:
# Join customer city and state for location features
gold_orders_order_items_customers = gold_orders_order_items.join(silver_customers.select("customer_id", "customer_city", "customer_state"), on = "customer_id", how = "left")
gold_orders_order_items_customers.count()

In [0]:
gold_orders_order_items_customers.printSchema()

In [0]:
# Load payment data and check available payment types
payments = spark.read.table("revenue_operations.silver.payments")
payments.select('payment_type').distinct().show()

In [0]:
from pyspark.sql.functions import lit, max as spark_max

# Aggregate payment data: sum installments, count records, one-hot encode payment types
silver_payments = payments.groupBy("order_id").agg(
    F.sum("payment_installments").alias("total_installments"),
    F.count("payment_type").alias("payment_record_count"),
    F.countDistinct("payment_type").alias("payment_type_count"),
    spark_max(F.when(F.col("payment_type") == "credit_card", 1).otherwise(0)).alias("credit_card"),
    spark_max(F.when(F.col("payment_type") == "debit_card", 1).otherwise(0)).alias("debit_card"),
    spark_max(F.when(F.col("payment_type") == "boleto", 1).otherwise(0)).alias("boleto"),
    spark_max(F.when(F.col("payment_type") == "voucher", 1).otherwise(0)).alias("voucher"),
    spark_max(F.when(F.col("payment_type") == "not_defined", 1).otherwise(0)).alias("not_defined")
)

print("Distinct order_ids in payments:", payments.select('order_id').distinct().count())
print("Rows in silver_payments:", silver_payments.count())
silver_payments.columns


In [0]:
# Create base order-level feature table with orders, items, customers, and payments
gold_orders_order_items_customers_payments = gold_orders_order_items_customers.join(silver_payments.select('order_id',
 'total_installments',
 'payment_record_count',
 'payment_type_count',
 'credit_card',
 'debit_card',
 'boleto',
 'voucher',
 'not_defined'), on = "order_id", how = "left")
gold_orders_order_items_customers_payments.printSchema()

In [0]:
gold_orders_order_items_customers_payments.count()

## 3. Product and Seller Feature Engineering

Aggregate item-level product and seller data to order level:
- Product dimensions: weight, volume metrics
- Item counts and distinct product counts
- Seller counts and geographic relationships
- Same-state seller flags and cross-state counts
- Earliest shipping limit date

In [0]:
# Load product dimensions for weight and volume features
products = spark.read.table("revenue_operations.silver.products")

In [0]:
# Load order items for aggregation
order_items = spark.read.table("revenue_operations.silver.order_items")

In [0]:
# Enrich order items with product dimensions (height, length, weight, width)
order_items_product_dims = order_items.join(products.select("product_id", "product_category_name", "product_height_cm", "product_length_cm", "product_weight_g", "product_width_cm"), on = "product_id", how = "left")
order_items_product_dims.printSchema()

In [0]:
# Load seller data and join to get seller city and state per item
sellers = spark.read.table("revenue_operations.silver.sellers")
order_items_product_dims_sellers = order_items_product_dims.join(sellers.select("seller_id", "seller_city", "seller_state"), on = "seller_id", how = "left")
order_items_product_dims_sellers.printSchema()

In [0]:
# Step 1: Join customer state to item level for state matching calculations
order_items_with_customer_state = order_items_product_dims_sellers.join(
    gold_orders_order_items_customers_payments.select("order_id", "customer_state"), 
    on="order_id", 
    how="left"
)

# Step 2: Aggregate item-level data to order-level

new_agg_seller_products = order_items_with_customer_state.groupBy("order_id").agg(
    # Product metrics: weights, volumes, and counts
    F.sum("product_weight_g").alias("total_product_weight"),
    F.avg("product_weight_g").alias("average_product_weight"),
    spark_max("product_weight_g").alias("max_product_weight"),
    F.sum(F.col("product_length_cm") * F.col("product_width_cm") * F.col("product_height_cm")).alias("total_product_volume"),
    F.count("*").alias("item_count"),
    F.countDistinct("product_id").alias("distinct_product_count"),
    
    # Seller count metrics
    F.countDistinct("seller_id").alias("seller_count"),
    F.countDistinct("seller_state").alias("seller_state_count"),
    
    # Seller-state flags: track if any/all sellers match customer state
    spark_max(F.when(F.col("seller_state") == F.col("customer_state"), 1).otherwise(0)).alias("any_seller_same_state_flag"),
    F.sum(F.when(F.col("seller_state") != F.col("customer_state"), 1).otherwise(0)).alias("cross_state_seller_count"),

    # Earliest shipping limit: the prediction point for late delivery
    F.min("shipping_limit_date").alias("earliest_shipping_limit_date")
)

# Step 3: Calculate flag indicating if ALL sellers are in customer's state
new_agg_seller_products = new_agg_seller_products.withColumn(
    "same_state_seller_flag",
    F.when((F.col("seller_count") - F.col("cross_state_seller_count")) == F.col("seller_count"), 1).otherwise(0)
)

print("Aggregated order count:", new_agg_seller_products.count())
new_agg_seller_products.printSchema()




In [0]:
# Join aggregated product/seller features to the base order-level table
agg_feature_extraction = gold_orders_order_items_customers_payments.join(new_agg_seller_products, on = "order_id", how = "left")
agg_feature_extraction.count()

In [0]:
display(agg_feature_extraction)

## 4. Distance Feature Engineering

Calculate geographic distances between customers and sellers:
- Load geolocation data (zip code to latitude/longitude)
- Join customer and seller coordinates to order items
- Apply Haversine formula for great-circle distance calculation
- Aggregate min, average, and max distances to order level

In [0]:
# Load geolocation lookup table (zip code to lat/lon)
geolocation = spark.read.table("revenue_operations.silver.geolocation")
display(geolocation)

In [0]:
# Join customer zip codes to geolocation for latitude/longitude
customer_geo = geolocation.join(silver_customers, silver_customers['customer_zip_code_prefix'] == geolocation['geolocation_zip_code_prefix'] , how = "left")

columns_to_keep = ["geolocation_avg_latitude", "geolocation_avg_longitude", "customer_id"]
customer_geo = customer_geo.select(*columns_to_keep)
customer_geo = customer_geo.withColumnRenamed("geolocation_avg_longitude", "customer_avg_longitude")
customer_geo = customer_geo.withColumnRenamed("geolocation_avg_latitude", "customer_avg_latitude")

# Join seller zip codes to geolocation for latitude/longitude
seller_geo = geolocation.join(sellers, sellers['seller_zip_code_prefix'] == geolocation["geolocation_zip_code_prefix"], how = "left")

columns_to_keep = ["geolocation_avg_latitude", "geolocation_avg_longitude", "seller_id"]
seller_geo = customer_seller_geo.select(*columns_to_keep)
seller_geo = seller_geo.withColumnRenamed("geolocation_avg_longitude", "seller_avg_longitude")
seller_geo = seller_geo.withColumnRenamed("geolocation_avg_latitude", "seller_avg_latitude")


In [0]:
# Join customer_id to items
order_items_product_dims_sellers = order_items_product_dims_sellers.join(gold_fact_orders.select("customer_id", "order_id"), on = "order_id", how = "left")

# Join customer and seller geolocation for distance calculation
order_items_product_dims_sellers = order_items_product_dims_sellers.join(customer_geo, on = "customer_id", how = "left")
order_items_product_dims_sellers = order_items_product_dims_sellers.join(seller_geo, on = "seller_id", how = "left")

In [0]:
# Haversine distance function: calculates great-circle distance between two lat/lon points
def haversine(df, lat1, lon1, lat2, lon2):
    # Earth's radius in kilometers
    R = 6371.0

    # Convert degrees to radians directly on DataFrame columns
    phi1 = F.radians(F.col(lat1))
    phi2 = F.radians(F.col(lat2))

    delta_phi = F.radians(F.col(lat2) - F.col(lat1))
    delta_lambda = F.radians(F.col(lon2) - F.col(lon1))

    # Haversine formula
    a = F.sin(delta_phi / 2)**2 + F.cos(phi1) * F.cos(phi2) * F.sin(delta_lambda / 2)**2
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

    # Return DataFrame with new column: distance in kilometers
    return df.withColumn("customer_seller_distance", R * c)

# Calculate the distance between customer and seller
order_items_product_dims_sellers = haversine(order_items_product_dims_sellers, "customer_avg_latitude", "customer_avg_longitude", "seller_avg_latitude", "seller_avg_longitude")
display(order_items_product_dims_sellers) 

In [0]:
# Aggregate customer-seller distances: min, avg, max per order
agg_distance = order_items_product_dims_sellers.groupBy("order_id").agg(
    F.min("customer_seller_distance").alias("min_customer_seller_distance"),
    F.avg("customer_seller_distance").alias("avg_customer_seller_distance"),
    F.max("customer_seller_distance").alias("max_customer_seller_distance")
)
display(agg_distance)
print("Number of rows in agg_distance : ", agg_distance.count())

In [0]:
# Add distance features to the main feature table
final_aggregated_features = agg_feature_extraction.join(agg_distance, on = "order_id", how = "left")
final_aggregated_features.printSchema()
print("Total rows in final_aggregated_features : ", final_aggregated_features.count())
display(final_aggregated_features)

## 5. Historical Seller Performance Features

Calculate each seller's historical late delivery rate using temporal window functions:
- Use only PAST orders (temporal filtering prevents data leakage)
- Window partitioned by seller, ordered by approval timestamp
- Exclude current and future orders from calculation
- Aggregate min, average, and max historical rates to order level

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: Get item-level data with order approval time and late flag
items_with_order_info = order_items_product_dims_sellers.select("order_id", "seller_id").join(final_aggregated_features.select("order_id", "order_approved_at", "late_delivery_flag"), on = "order_id", how = "inner")

# Step 2: Define temporal window - only look at PREVIOUS orders for each seller
# This prevents data leakage by excluding current and future orders
window = Window.partitionBy("seller_id").orderBy("order_approved_at").rowsBetween(Window.unboundedPreceding, -1)

# Step 3: Calculate seller's historical late rate at item level
items_with_historical_rate = items_with_order_info.withColumn(
    "seller_historical_late_rate",
    F.avg(
        F.when(F.col("late_delivery_flag") == 'late', 1).otherwise(0)
    ).over(window)
)

# Step 4: Aggregate to order level (min/avg/max across sellers)
order_historical_seller_late_rate = items_with_historical_rate.groupBy("order_id").agg(
    F.min("seller_historical_late_rate").alias("min_seller_historical_late_rate"),
    F.avg("seller_historical_late_rate").alias("avg_seller_historical_late_rate"),
    F.max("seller_historical_late_rate").alias("max_seller_historical_late_rate")
)

# Step 5: Join historical rates back to main feature table
final_features_with_historical = final_aggregated_features.join(order_historical_seller_late_rate, on = "order_id", how = "left")

print("Final row count: ", final_features_with_historical.count())
final_features_with_historical.printSchema()

In [0]:
display(final_features_with_historical)

## 6. Feature Selection and Cleanup

Remove columns that are not available at prediction time:
- Identifier columns: order_id, customer_id
- Leakage features: delivery_date, delivery_days, average_review_score
- Only keep features available before shipping deadline

In [0]:
# Drop identifier columns and leakage features (known only after delivery)
columns_to_drop = ['order_id',
 'customer_id',
 'order_status',
 'purchase_date',
 'delivery_date',
 'delivery_days',
 'average_review_score',
 'number_of_payment_records']
final_features = final_features_with_historical.drop(*columns_to_drop)

In [0]:
final_features.columns

## 7. Final Validation and Quality Checks

Comprehensive validation to ensure dataset is ready for ML modeling:
- Row and column counts verification
- Leakage detection (confirm no future information)
- Target variable distribution and class balance
- Null value analysis by column
- Required feature presence check
- Summary statistics for key numeric features

In [0]:
# ============================================
# FINAL FEATURE TABLE VALIDATION
# ============================================

print("=" * 80)
print("1. BASIC COUNTS")
print("=" * 80)
row_count = final_features.count()
col_count = len(final_features.columns)
print(f"✓ Total Rows: {row_count:,} (Expected: 96,462)")
print(f"✓ Total Columns: {col_count} (Expected: 40 = 1 target + 39 features)")
print()

# ============================================
print("=" * 80)
print("2. LEAKAGE CHECK - Verify NO leakage columns present")
print("=" * 80)
leakage_columns = ['delivery_date', 'delivery_days', 'average_review_score', 'order_id', 'customer_id']
present_leakage = [col for col in leakage_columns if col in final_features.columns]

if present_leakage:
    print(f"❌ LEAKAGE DETECTED! These columns should NOT be present: {present_leakage}")
else:
    print("✓ PASSED: No leakage columns detected")
print()

# ============================================
print("=" * 80)
print("3. TARGET VARIABLE CHECK")
print("=" * 80)
if 'late_delivery_flag' in final_features.columns:
    print("✓ Target column 'late_delivery_flag' is present")
    target_dist = final_features.groupBy('late_delivery_flag').count().orderBy('late_delivery_flag')
    print("\nTarget Distribution:")
    target_dist.show()
    
    # Calculate class balance
    total = final_features.count()
    late_count = final_features.filter(F.col('late_delivery_flag') == 'late').count()
    on_time_count = final_features.filter(F.col('late_delivery_flag') == 'on_time').count()
    print(f"  Late deliveries: {late_count:,} ({late_count/total*100:.1f}%)")
    print(f"  On-time deliveries: {on_time_count:,} ({on_time_count/total*100:.1f}%)")
else:
    print("❌ ERROR: Target column 'late_delivery_flag' is missing!")
print()

# ============================================
print("=" * 80)
print("4. NULL VALUE CHECK")
print("=" * 80)
print("Checking for null counts in each column...\n")
null_counts = final_features.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in final_features.columns]
).collect()[0].asDict()

high_null_cols = {col: count for col, count in null_counts.items() if count > 0}

if high_null_cols:
    print(f"⚠️  Columns with NULL values (total: {len(high_null_cols)}):")
    for col, count in sorted(high_null_cols.items(), key=lambda x: x[1], reverse=True):
        pct = (count / row_count) * 100
        print(f"  - {col}: {count:,} nulls ({pct:.2f}%)")
else:
    print("✓ PASSED: No null values found in any column")
print()

# ============================================
print("=" * 80)
print("5. REQUIRED FEATURES CHECK")
print("=" * 80)
required_features = [
    'order_approved_at', 'estimated_delivery_date', 'late_delivery_flag',
    'customer_state', 'total_payment_value',
    'seller_count', 'min_customer_seller_distance', 'avg_customer_seller_distance',
    'min_seller_historical_late_rate', 'avg_seller_historical_late_rate'
]

missing_features = [feat for feat in required_features if feat not in final_features.columns]

if missing_features:
    print(f"❌ ERROR: Missing required features: {missing_features}")
else:
    print("✓ PASSED: All key required features are present")
print()

# ============================================
print("=" * 80)
print("6. FEATURE SUMMARY STATISTICS")
print("=" * 80)
print("Numeric feature ranges (sample):")
numeric_cols = ['total_payment_value', 'seller_count', 'item_count', 
                'min_customer_seller_distance', 'avg_customer_seller_distance']

for col in numeric_cols:
    if col in final_features.columns:
        stats = final_features.select(
            F.min(col).alias('min'),
            F.max(col).alias('max'),
            F.avg(col).alias('avg')
        ).collect()[0]
        print(f"  {col}:")
        print(f"    Min: {stats['min']}, Max: {stats['max']}, Avg: {stats['avg']:.2f}")
print()

# ============================================
print("=" * 80)
print("7. FINAL VALIDATION SUMMARY")
print("=" * 80)

validation_passed = (
    row_count == 96462 and
    col_count == 40 and
    len(present_leakage) == 0 and
    'late_delivery_flag' in final_features.columns and
    len(missing_features) == 0
)

if validation_passed:
    print("✅ ALL CHECKS PASSED! Dataset is ready for ML modeling.")
else:
    print("⚠️  SOME CHECKS FAILED. Review the validation results above.")
    
print("=" * 80)